# Baseline EcomGPT-7B × A (LoRA, 4M token / 500 step)


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'no GPU'
p = torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory / 1e9, 1), 'GB')
assert p.total_memory > 40e9, 'cần GPU >= 40GB (RTX Pro 6000); kernel bị cấp GPU nhỏ'
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True)
try:
    import peft  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft'], check=True)

LOCAL = Path('/kaggle/input/ecomgpt-7b')
if not (LOCAL / 'pytorch_model.bin.index.json').exists():
    hits = list(Path('/kaggle/input').rglob('pytorch_model.bin.index.json'))
    LOCAL = hits[0].parent if hits else LOCAL
if (LOCAL / 'pytorch_model.bin.index.json').exists():
    MODEL_DIR = str(LOCAL)
    print('local model', MODEL_DIR)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'modelscope'], check=True)
    from modelscope import snapshot_download
    MODEL_DIR = snapshot_download('damo/nlp_ecomgpt_multilingual-7B-ecom',
                                  cache_dir='/kaggle/working/ms')
    print('downloaded', MODEL_DIR)


In [ ]:
import json, re, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

DATASET = 'B'
CONFIG = 'balanced'
BUDGET = 4_000_000

DATA = Path('/kaggle/input/tiki-matrix-data')
if not (DATA / 'quota_sampling.py').exists():
    hits = sorted(Path('/kaggle/input').rglob('quota_sampling.py'))
    if not hits:
        raise SystemExit('dataset not mounted: ' + str(list(Path('/kaggle/input').iterdir())))
    DATA = hits[0].parent
sys.path.insert(0, str(DATA))
from quota_sampling import quota_indices
print('DATA', DATA)

units = pd.read_parquet(DATA / f'units_{DATASET}.parquet')
meta = json.loads((DATA / f'meta_{DATASET}.json').read_text(encoding='utf-8'))
cfg = meta['configs'].get(CONFIG)
idx, actual = quota_indices(units, cfg, 42, BUDGET)
sample = units.iloc[idx].reset_index(drop=True)
eval_frame = pd.read_parquet(DATA / f'eval_{DATASET}.parquet')
print('sample', len(sample), 'tokens', round(sample.eff_tokens.sum()/1e6, 3), actual)
print('eval', len(eval_frame))
sample[['pair_id','group','eff_tokens']].to_parquet('/kaggle/working/sample_ecomgpt_b_seed42.parquet', index=False)
train_tokens = float(units.eff_tokens.sum())
print('epochs_seen', round(float(sample.eff_tokens.sum())/train_tokens, 4))

CJK = re.compile(r'[\u4e00-\u9fff]')
PUNCT = re.compile(r'([,.;:!?()\[\]{}"“”‘’、，。；：！？（）【】《》])')

def tok_vi(text: str) -> str:
    t = str(text or '').strip()
    t = PUNCT.sub(r' \1 ', t)
    return re.sub(r'\s+', ' ', t).strip()

def tok_zh(text: str) -> str:
    t = str(text or '').strip()
    t = PUNCT.sub(r' \1 ', t)
    t = re.sub(r'([\u4e00-\u9fff])', r' \1 ', t)
    return re.sub(r'\s+', ' ', t).strip()

WORK = Path('/kaggle/working/work')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'train.vi').write_text('\n'.join(tok_vi(x) for x in sample.source_text) + '\n', encoding='utf-8')
(WORK / 'train.zh').write_text('\n'.join(tok_zh(x) for x in sample.target_text) + '\n', encoding='utf-8')
(WORK / 'eval.vi').write_text('\n'.join(tok_vi(x) for x in eval_frame.source) + '\n', encoding='utf-8')
print('wrote train/eval text')


In [ ]:
import math
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

INSTRUCTION = 'Translate the following Vietnamese e-commerce text into Simplified Chinese. Return only the translation.'
TEMPLATE = ('Below is an instruction that describes a task, paired with an input that provides further context. '
            'Write a response that appropriately completes the request.\n\n'
            '### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n')

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16,
                                             trust_remote_code=True, low_cpu_mem_usage=True)
model.gradient_checkpointing_enable(); model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                         bias='none', task_type='CAUSAL_LM',
                                         target_modules='all-linear'))
model.to('cuda').train(); model.config.use_cache = False
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=1e-4)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s+1)/20) * max(0.0, 0.5*(1+math.cos(math.pi*min(1.0, s/500)))))

MAXLEN = 960
TOKENS_PER_STEP = 8000.0
MICRO = 2

def encode(row):
    p = TEMPLATE.format(instruction=INSTRUCTION, input=str(row['source_text']))
    pid = tokenizer(p, add_special_tokens=False).input_ids
    if len(pid) >= MAXLEN - 8:
        return None
    full = tokenizer(p + str(row['target_text']) + tokenizer.eos_token,
                     add_special_tokens=False, truncation=True, max_length=MAXLEN).input_ids
    if len(full) <= len(pid) + 1:
        return None
    return full, [-100]*len(pid) + full[len(pid):]

def micro_batches(rows, steps=500):
    plan, cur, cur_tok = [], [], 0.0
    for _, row in rows.iterrows():
        item = encode(row)
        if item is None:
            continue
        cur.append(item)
        cur_tok += len(item[0])
        if cur_tok >= TOKENS_PER_STEP and len(plan) < steps - 1:
            plan.append(cur); cur, cur_tok = [], 0.0
    if cur:
        plan.append(cur)
    while len(plan) < steps:
        last = plan.pop()
        if len(last) <= 1:
            plan.extend([last, last]); continue
        mid = len(last) // 2
        plan.extend([last[:mid], last[mid:]])
    return plan[:steps]

plan = micro_batches(sample)
print('steps', len(plan), 'avg rows/step', sum(len(s) for s in plan)/len(plan), flush=True)
for step, items in enumerate(plan):
    opt.zero_grad(set_to_none=True)
    step_tokens = sum(len(x[0]) for x in items)
    for start in range(0, len(items), MICRO):
        chunk = items[start:start+MICRO]
        L = max(len(x[0]) for x in chunk)
        ids = torch.tensor([x[0] + [tokenizer.pad_token_id]*(L-len(x[0])) for x in chunk]).cuda()
        lab = torch.tensor([x[1] + [-100]*(L-len(x[1])) for x in chunk]).cuda()
        mask = torch.tensor([[1]*len(x[0]) + [0]*(L-len(x[0])) for x in chunk]).cuda()
        out = model(input_ids=ids, labels=lab, attention_mask=mask)
        n_tok = int((lab != -100).sum().item())
        loss = out.loss * (n_tok / max(1.0, step_tokens))
        loss.backward()
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    opt.step(); sched.step()
    if step % 25 == 0:
        print(step, round(float(out.loss), 4), flush=True)
model.save_pretrained('/kaggle/working/ecom_adapter')


In [ ]:
import pandas as pd
ev = pd.read_parquet(DATA / 'eval_A.parquet')
for dsname in ('A', 'B'):
    U = pd.read_parquet(DATA / f'units_{dsname}.parquet')
    overlap = set(ev.product_id) & set(U.product_id)
    tok = float(U.eff_tokens.sum())
    print(dsname, 'rows', len(U), 'tokensM', round(tok/1e6, 3),
          'overlap', len(overlap), 'epochs@4M', round(4e6/tok, 3))
    assert not overlap, f'LEAK {dsname}'
print('LEAK CHECK OK')


In [ ]:
model.eval(); model.gradient_checkpointing_disable(); model.config.use_cache = True
import re
def build(row):
    return TEMPLATE.format(instruction=INSTRUCTION, input=str(row['source']))

pred = {}
for max_new, part in eval_frame.groupby('max_new'):
    print('generate max_new', max_new, 'rows', len(part), flush=True)
    for start in range(0, len(part), 8):
        batch = part.iloc[start:start+8]
        enc = tokenizer([build(r) for _, r in batch.iterrows()], return_tensors='pt',
                        padding=True, truncation=True, max_length=1024).to('cuda')
        with torch.inference_mode():
            gen = model.generate(**enc, max_new_tokens=int(max_new), do_sample=False,
                                 eos_token_id=tokenizer.eos_token_id,
                                 pad_token_id=tokenizer.pad_token_id)
        texts = tokenizer.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        for eid, text in zip(batch.eval_id, texts):
            pred[eid] = text.strip().split('###')[0].strip()
        if start % 160 == 0:
            print(start, '/', len(part), flush=True)

outs = [pred.get(eid, '') for eid in eval_frame.eval_id]
Path('/kaggle/working/work/pred.zh').write_text('\n'.join(outs) + '\n', encoding='utf-8')
print('done', len(outs))


In [ ]:
import pandas as pd

def detok(text: str) -> str:
    t = re.sub(r'\s+', ' ', str(text or '')).strip()
    t = re.sub(r'(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])', '', t)
    t = re.sub(r'\s+(?=[，。！？；：、）】》])', '', t)
    t = re.sub(r'(?<=[（【《])\s+', '', t)
    return t

pred_path = Path('/kaggle/working/work/pred.zh')
hyps = [l.rstrip('\n') for l in pred_path.read_text(encoding='utf-8').splitlines()]
if len(hyps) < len(eval_frame):
    hyps += [''] * (len(eval_frame) - len(hyps))
out = eval_frame[['eval_id', 'dataset', 'pair_type', 'source', 'reference']].copy()
out['hypothesis'] = [detok(h) for h in hyps[:len(out)]]
out['config'] = 'ecomgpt_' + DATASET.lower()
out['seed'] = 42
out.to_parquet(f'/kaggle/working/hyp_{out.config.iloc[0]}_seed42.parquet', index=False)
print(out.head(3).to_string()[:500])
print('empty', float((out.hypothesis.str.len() == 0).mean()), 'n', len(out))
